In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv("E:\\DS\\tezendra\\Projects\\ML Models_Choosing By Problem\\healthcare_data_10000.csv")

# Encode target: Low = 0, High = 1
df['health_risk_category'] = df['health_risk_category'].map({'Low': 0, 'High': 1})

# Features and target
X = df.drop(columns=['health_risk_category'])
y = df['health_risk_category']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

model = XGBClassifier(n_estimators = 100, learning_rate = 0.1, eval_metric = 'logloss', random_state = 42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

classification_report = classification_report(y_pred, y_test, target_names= ['Low', "High"])
print(classification_report)

              precision    recall  f1-score   support

         Low       1.00      1.00      1.00      1736
        High       1.00      0.99      1.00       264

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [13]:
pip install scikit-optimize


   ---------------------------------------- 0/2 [pyaml]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   -------------------- ------------------- 1/2 [scikit-optimize]
   ---------------------------------------- 2/2 [scikit-optimize]

Note: you may need to restart the kernel to use updated packages.


In [14]:
from skopt import BayesSearchCV

In [15]:
search_space = {
    'n_estimators': (100, 1000,10),
    'max_depth': (3, 10),
    'learning_rate': (0.01, 0.3),
    'subsample': (0.5, 1.0),
    'colsample_bytree': (0.5, 1.0),
    'gamma': (0, 5),
    'min_child_weight': (1, 10,2)}

In [17]:
xgb = XGBClassifier(random_state = 42)
opt = BayesSearchCV(
    estimator=xgb,
    search_spaces=search_space,
    n_iter=32,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

In [19]:
opt.fit(X_train, y_train)
print("Best parameters found: ", opt.best_params_)

Best parameters found:  OrderedDict({'colsample_bytree': 0.5, 'gamma': 0, 'learning_rate': 0.09041190958730318, 'max_depth': 3, 'min_child_weight': 10, 'n_estimators': 1000, 'subsample': 0.8860275071142114})


In [20]:
best_model = opt.best_estimator_
y_pred_best = best_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred_best)
print("Mean Squared Error of the best model: ", mse)
rmse = np.sqrt(mse)
print("Root Mean Squared Error of the best model: ", rmse)

Mean Squared Error of the best model:  0.001
Root Mean Squared Error of the best model:  0.03162277660168379
